# Supervised Learning. Decision Trees and ensembles

## 1.Downloading and preprocessing data

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from category_encoders.count import CountEncoder

In [ ]:
data = pd.read_csv("data/training.csv", parse_dates=[2])

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   RefId                              72983 non-null  int64         
 1   IsBadBuy                           72983 non-null  int64         
 2   PurchDate                          72983 non-null  datetime64[ns]
 3   Auction                            72983 non-null  object        
 4   VehYear                            72983 non-null  int64         
 5   VehicleAge                         72983 non-null  int64         
 6   Make                               72983 non-null  object        
 7   Model                              72983 non-null  object        
 8   Trim                               70623 non-null  object        
 9   SubModel                           72975 non-null  object        
 10  Color                             

In [ ]:
data = data.drop(["PRIMEUNIT", "AUCGUART"], axis=1)

In [ ]:
def data_splitter(data, field, train, val, test, seed=21):
    np.random.seed(seed)
    base_len = len(data)
    data = data.sort_values(field, ascending=True)

    train_in = data.loc[data[field] < data.iloc[int(base_len * train)][field]]
    train_in = train_in.index.to_list()
    data = data[~data.index.isin(train_in)]

    val_in = data.loc[data[field] < data.iloc[int(base_len * val)][field]]
    val_in = val_in.index.to_list()
    test_in = data[~data.index.isin(val_in)].index.to_list()

    np.random.shuffle(train_in)
    np.random.shuffle(test_in)
    np.random.shuffle(val_in)

    return train_in, val_in, test_in

In [ ]:
data = data.astype({"PurchDate":"object"})
train, val, test = data_splitter(data, "PurchDate", 1/3, 1/3, 1/3)
train = data.iloc[train]
val = data.iloc[val]
test = data.iloc[test]

In [ ]:
objs = data.select_dtypes(include='object').columns
encoder = CountEncoder(cols=list(objs)).fit(train)
train = encoder.transform(train)
test = encoder.transform(test)
val = encoder.transform(val)

In [ ]:
means = train.mean()
train = train.fillna(means)
val = val.fillna(means)
test = test.fillna(means)

In [ ]:
scaler = MinMaxScaler().fit(train)
train = pd.DataFrame(data=scaler.transform(train), columns=train.columns)
test = pd.DataFrame(data=scaler.transform(test), columns=train.columns)
val = pd.DataFrame(data=scaler.transform(val), columns=train.columns)

In [ ]:
def gini_score(scores, y_true):
    func = 2 * roc_auc_score(y_true, scores) - 1
    return abs(func)

## 2.Decision Tree Classifier and Regressor

In [ ]:
class Node():
    def __init__(self, depth, feat, thr, score, prob):
        self.right = None
        self.left = None
        self.depth = depth
        self.feat = feat
        self.thr = thr
        self.score = score
        self.prob = prob
        self.mask = None

def gini(y):
    p = np.mean(y)

    return 1 - (p**2 + (1-p)**2)

def mse(y):
    y_mean = y.mean()
    sq_diff = np.vectorize(lambda x: (x - y_mean) ** 2)

    return (sq_diff(y).sum()) / len(y)

def prob_cl(y):
    return sum(y) / len(y)

def prob_reg(y):
    return y.mean()

In [ ]:
def clas_split(parent, X, y, max_feat, min_els, data_ratio=1.0):
    if len(X) == 1:
        return [None, None, 0, 0, y[0]]

    feats, samps = len(X[0]), len(X[1])
    base = gini(y)
    res = [None, None, -np.inf, None, 0]

    if base == 0:
        return res

    if max_feat > len(X[0]):
        max_feat = len(X[0])

    np.random.seed(np.random.randint(1, 100))
    feat_batch = np.random.choice(feats, size=max_feat, replace=False)

    for feat in feat_batch:
        sort_id = np.argsort(X[:, feat])
        X_sort = X[sort_id, feat]
        y_sort = y[sort_id]

        cls_diff = np.where(y_sort[:-1] != y_sort[1:])[0]
        thrs = (X_sort[cls_diff] + X_sort[cls_diff + 1]) / 2

        if data_ratio < 1 and len(thrs) > 1:
            thr_count = round(len(thrs) * data_ratio)
            thrs = np.random.choice(thrs, size=thr_count)

        for thr in thrs:
            mask = X[:, feat] <= thr
            y_split = [y[mask], y[~mask]]

            if len(y_split[1]) == 0:
                continue

            if len(y_split[0]) < min_els or len(y_split[1]) < min_els:
                continue

            metr = [gini(y_split[0]), gini(y_split[1])]
            weight = [len(y_split[0]) / samps, len(y_split[1]) / samps]
            metr_gain = base - (weight[0] * metr[0] + weight[1] * metr[1])

            if metr_gain < res[2]:
                continue

            right = Node(parent.depth + 1, None, None, gini(y_split[1]), prob_cl(y_split[1]))
            left = Node(parent.depth + 1, None, None, gini(y_split[0]), prob_cl(y_split[0]))

            left.mask, right.mask = mask, ~mask

            res = [left, right, metr_gain, feat, thr]

    return res


In [ ]:
def reg_split(parent, X, y, max_feat, min_els, data_ratio=1.0):
    if len(X) == 1:
        return [None, None, 0, 0, y[0]]

    feats, samps = len(X[0]), len(X[1])
    base = mse(y)
    res = [None, None, -np.inf, None, 0]

    if base == 0:
        return res

    if max_feat > len(X[0]):
        max_feat = len(X[0])

    np.random.seed(np.random.randint(1, 100))
    feat_batch = np.random.choice(feats, size=max_feat, replace=False)

    for feat in feat_batch:
        sort_id = np.argsort(X[:, feat])
        X_sort = X[sort_id, feat]
        y_sort = y[sort_id]

        cls_diff = np.where(y_sort[:-1] != y_sort[1:])[0]
        thrs = (X_sort[cls_diff] + X_sort[cls_diff + 1]) / 2

        if data_ratio < 1 and len(thrs) > 1:
            thr_count = round(len(thrs) * data_ratio)
            thrs = np.random.choice(thrs, size=thr_count)

        y_cumsum = np.cumsum(y_sort)
        y_cumsum_sq = np.cumsum(y_sort ** 2)

        for i, thr in enumerate(thrs):
            split_id = cls_diff[i] + 1

            if split_id < min_els or (samps - split_id) < min_els:
                continue

            left_size = split_id
            right_size = samps - split_id

            left_sum = y_cumsum[split_id - 1]
            right_sum = y_cumsum[-1] - left_sum

            left_sum_sq = y_cumsum_sq[split_id - 1]
            right_sum_sq = y_cumsum_sq[-1] - left_sum_sq

            mse_left = (left_sum_sq - left_sum ** 2 / left_size) / left_size
            mse_right = (right_sum_sq - right_sum ** 2 / right_size) / right_size

            metr_gain = base - (left_size/samps * mse_left + right_size/samps * mse_right)

            if metr_gain < res[2]:
                continue

            mask = X[:, feat] <= thr

            if sum(mask) == 0 or sum(~mask) == 0:
                continue

            right = Node(parent.depth + 1, None, None, mse_right, right_sum / right_size)
            left = Node(parent.depth + 1, None, None, mse_left, left_sum / left_size)

            left.mask, right.mask = mask, ~mask

            res = [left, right, metr_gain, feat, thr]

    return res

In [ ]:
class DecisTree():
    def __init__(self, max_depth=5, max_features=10, min_els=5, data_ratio=1.0):
        self.max_depth = max_depth
        self.root = None
        self.max_features = max_features
        self.min_els = min_els
        self.data_ratio = data_ratio

    def predict_proba(self, X):
        X = np.array(X)
        res = []

        for row in X:
            res.append(self.get_prob(row, self.root))

        return np.array(res)

    def get_prob(self, row, node):
        if node.left is None:
            return node.prob

        if row[node.feat] <= node.thr:
            return self.get_prob(row, node.left)

        return self.get_prob(row, node.right)

In [ ]:
class DecisTreeCl(DecisTree):
    def __init__(self, max_depth=5, max_features=10, min_els=5, data_ratio=1.0):
         super().__init__(max_depth, max_features, min_els, data_ratio)

    def fit(self, X, y, random_state=21):
        X = np.array(X)
        y = np.array(y)
        np.random.seed(random_state)

        self.root = Node(0, None, None, gini(y), prob_cl(y))
        self.node_creat(self.root, X, y)

    def node_creat(self, parent, X, y):
        if parent.depth == self.max_depth:
            return 0

        res = clas_split(parent, X, y, self.max_features, self.min_els,
                              data_ratio=self.data_ratio)

        parent.left, parent.right, _, parent.feat, parent.thr = res

        if parent.left is None or parent.right is None:
            return 0

        self.node_creat(parent.left, X[parent.left.mask], y[parent.left.mask])
        self.node_creat(parent.right, X[parent.right.mask], y[parent.right.mask])

    def predict(self, X, threshold=0.5):
        thr = np.vectorize(lambda x: 1 if x > threshold else 0)
        return thr(self.predict_proba(X))


In [ ]:
class DecisTreeReg(DecisTree):
    def __init__(self, max_depth=5, max_features=10, min_els=5, data_ratio=1.0):
        super().__init__(max_depth, max_features, min_els, data_ratio)

    def fit(self, X, y, random_state=21):
        X = np.array(X)
        y = np.array(y)
        np.random.seed(random_state)

        self.root = Node(0, None, None, mse(y), prob_reg(y))
        self.node_creat(self.root, X, y)

    def node_creat(self, parent, X, y):
        if parent.depth == self.max_depth:
            return 0

        res = reg_split(parent, X, y, self.max_features, self.min_els,
                              data_ratio=self.data_ratio)

        parent.left, parent.right, _, parent.feat, parent.thr = res

        if parent.left is None or parent.right is None:
            return 0

        self.node_creat(parent.left, X[parent.left.mask], y[parent.left.mask])
        self.node_creat(parent.right, X[parent.right.mask], y[parent.right.mask])

    def predict(self, X):
        return self.predict_proba(X)

## 3.Model score evaluation (my implementation)

In [ ]:
%%time

tree = DecisTreeCl(25, 10)
tree.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"])

CPU times: user 1min 21s, sys: 120 ms, total: 1min 21s
Wall time: 1min 39s


In [ ]:
res = tree.predict_proba(val.drop("IsBadBuy", axis=1))
print(gini_score(res, val["IsBadBuy"]))

0.22077879109792597


In [ ]:
%%time

tree = DecisTreeCl(25, 10, data_ratio=0.6)
tree.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"])

CPU times: user 46.9 s, sys: 54.4 ms, total: 46.9 s
Wall time: 48.8 s


In [ ]:
res = tree.predict_proba(val.drop("IsBadBuy", axis=1))
print(gini_score(res, val["IsBadBuy"]))

0.25484957761197347


## 4.Model score evaluation (sklearn's implementation)

In [ ]:
sk_tree = DecisionTreeClassifier(max_depth=25, random_state=21, max_features=10)
sk_tree.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"])
sk_res = sk_tree.predict_proba(val.drop("IsBadBuy", axis=1))

In [ ]:
print(gini_score(sk_res[:, 1], val["IsBadBuy"]))

0.13467732017465872


My model performs somewhat better than sklearn's. But the results of my model are still far from good. The Extra Tree works a bit better than the normal one and a bit faster. The reason for which lies in the fact that it gets to go through subsistanly less thresholds making the model simpler and hence better fit for the new data.

## 5.Random forest

In [ ]:
class RandomForest():
    def __init__(self, max_depth=10, num_of_trees=10, max_features=10, min_els=1):
        self.max_depth = max_depth
        self.num_of_trees = num_of_trees
        self.max_features = max_features
        self.min_els = min_els
        self.trees = []

    def fit(self, X, y, data_rat=1.0, random_state=21):
        X = np.array(X)
        y = np.array(y)
        batch_size = round(len(X) * data_rat)

        np.random.seed(random_state)
        random_states = np.random.randint(0, 100, self.num_of_trees)

        for i in range(self.num_of_trees):
            np.random.seed(random_states[i])

            batch_map = np.random.choice(range(len(X)), size=batch_size, replace=False)
            batch_X, batch_y = X[batch_map], y[batch_map]

            tree = DecisTreeCl(self.max_depth, self.max_features, self.min_els)
            tree.fit(batch_X, batch_y)

            self.trees.append(tree)

    def predict_proba(self, X):
        X = np.array(X)
        probas = np.zeros(len(X))

        for tree in self.trees:
            proba = tree.predict_proba(X)
            probas += proba

        return probas / self.num_of_trees

    def predict(self, X, threshold=0.5):
        thr = np.vectorize(lambda x: 1 if x >= threshold else 0)
        return thr(self.predict_proba(X))


In [ ]:
%%time

forest = RandomForest(10, 10)
forest.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"])
for_res = forest.predict_proba(val.drop("IsBadBuy", axis=1))

CPU times: user 12min 22s, sys: 759 ms, total: 12min 23s
Wall time: 12min 39s


In [ ]:
print(gini_score(for_res, val["IsBadBuy"]))

0.2840087155436153


In [ ]:
%%time

forest.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"], data_rat=0.6)
for_res = forest.predict_proba(val.drop("IsBadBuy", axis=1))

CPU times: user 4min 46s, sys: 304 ms, total: 4min 46s
Wall time: 4min 58s


In [ ]:
print(gini_score(for_res, val["IsBadBuy"]))

0.41077987468810684


## 6.GBDT

In [ ]:
class GBDT():
    def __init__(self, max_depth=7, number_of_trees=10, max_features=10,
                 rate=0.1, min_els=1, data_ratio=1.0):
        self.max_depth = max_depth
        self.num_of_trees = number_of_trees
        self.max_features = max_features
        self.rate = rate
        self.min_els = min_els
        self.data_ratio = data_ratio
        self.trees = []

    def fit(self, X, y, random_state=21):
        X = np.array(X)
        y = np.array(y)

        F = np.zeros(len(y))
        p = self.sigmoid(F)

        np.random.seed(random_state)
        random_states = np.random.randint(0, 100, self.num_of_trees)

        for i in range(self.num_of_trees):
            np.random.seed(random_states[i])

            grad = y - p
            tree = DecisTreeReg(self.max_depth, self.max_features,
                                self.min_els, self.data_ratio)
            tree.fit(X, grad, random_states[i])

            self.trees.append(tree)

            F += self.rate * tree.predict(X)
            p = self.sigmoid(F)

    def sigmoid(self, x):
        x = np.clip(x, -500, 500)
        return 1 / (1 + np.exp(-x))

    def predict_proba(self, X):
        X = np.array(X)

        F = np.zeros(len(X))
        for i in range(self.num_of_trees):
            F += self.rate * self.trees[i].predict(X)

        p = self.sigmoid(F)

        return p

    def predict(self, X, threshold=0.5):
        thr = np.vectorize(lambda x: 1 if x > threshold else 0)
        return thr(self.predict_proba(X))


In [ ]:
%%time

boost = GBDT(10, 10, rate=1)
boost.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"])
boost_res = boost.predict_proba(val.drop("IsBadBuy", axis=1))

CPU times: user 28 s, sys: 195 ms, total: 28.2 s
Wall time: 28.5 s


In [ ]:
print(gini_score(boost_res, val["IsBadBuy"]))

0.103747726484265


## 7.LightGBM, Catboost and XGBoost

In [ ]:
%%capture

lgb = LGBMClassifier(max_depth=10, n_estimators=5, boosting_type="dart")
lgb.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"])
lgb_res = lgb.predict_proba(val.drop("IsBadBuy", axis=1))

In [ ]:
print(gini_score(lgb_res[:, 1], val["IsBadBuy"]))

0.4664562938971004


In [ ]:
%%capture

for col in data.select_dtypes(include=['object']).columns:
    if data[col].isnull().any():
        most_frequent = data[col].mode()
        if len(most_frequent) > 0:
            data[col].fillna(most_frequent[0], inplace=True)

cat_train, cat_val, cat_test = data_splitter(data, "PurchDate", 1/3, 1/3, 1/3)
data_ntime = data.drop(["PurchDate"], axis=1)
cat_train = data_ntime.iloc[cat_train]
cat_val = data_ntime.iloc[cat_val]
cat_test = data_ntime.iloc[cat_test]

cat_feats = ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'VNST']
cbt = CatBoostClassifier(max_depth=6, n_estimators=100, cat_features=cat_feats)
cbt.fit(cat_train.drop("IsBadBuy", axis=1), cat_train["IsBadBuy"])
cbt_res = cbt.predict_proba(cat_val.drop("IsBadBuy", axis=1))


In [ ]:
print(gini_score(cbt_res[:, 1], val["IsBadBuy"]))

0.4388808045083852


In [ ]:
xgb = XGBClassifier(max_depth=6, n_estimators=100, eval_metric='aucpr')
xgb.fit(train.drop("IsBadBuy", axis=1), train["IsBadBuy"])
xgb_res = xgb.predict_proba(val.drop("IsBadBuy", axis=1))

In [ ]:
print(gini_score(xgb_res[:, 1], val["IsBadBuy"]))

0.421802478391313


**Catboost** works with categorical features using the method called Oredered Target Statistics. The benefits of this method are that they prevent target leakage. It achieves this by: permutating feature, and then calculating the target statistics for each categorical value.  

**DART** method in XGBoost is the method of training which makes GBTrees simpler and better suited for production. It does this by randomly dropping trees during fitting process. This process helps trees generalise better, and hence be more suitable for real world applications.

The model which performed the best is LightGBM. Other models performed similarly well and as such the gap between the best model and the others is almost nonexistent. The Catboost algorithm performs fairly good because of the amount of categorical features. And the worst on to perform is XGBoost, probaly because it doesn't have a good way of handling categorical features.

## 8.Best model estimation

In [ ]:
forest_res = forest.predict_proba(train.drop("IsBadBuy", axis=1))
print("train -", gini_score(forest_res, train["IsBadBuy"]))
forest_res = forest.predict_proba(val.drop("IsBadBuy", axis=1))
print("valid -", gini_score(forest_res, val["IsBadBuy"]))
forest_res = forest.predict_proba(test.drop("IsBadBuy", axis=1))
print("test  -", gini_score(forest_res, test["IsBadBuy"]))

train - 0.7328464363075557
valid - 0.41077987468810684
test  - 0.40846029006363094


From the results we see we can either say that model is overfitted because there is a significant drop from the train to the validation score, but on the over hand we can also state that it is not overfitted, because the valid and test scores are equal. In my opinion the model is not overfitted, because the drop from the train to valid is actually expected especially in Decision Tree algorithms.

In [ ]:
lgb_res = lgb.predict_proba(train.drop("IsBadBuy", axis=1))
print("train -", gini_score(lgb_res[:, 1], train["IsBadBuy"]))
lgb_res = lgb.predict_proba(val.drop("IsBadBuy", axis=1))
print("valid -", gini_score(lgb_res[:, 1], val["IsBadBuy"]))
lgb_res = lgb.predict_proba(test.drop("IsBadBuy", axis=1))
print("test  -", gini_score(lgb_res[:, 1], test["IsBadBuy"]))

train - 0.5470934094692226
valid - 0.4664562938971004
test  - 0.4787986874226704


In [ ]:
res = tree.predict_proba(train.drop("IsBadBuy", axis=1))
print("train -", gini_score(res, train["IsBadBuy"]))
res = tree.predict_proba(val.drop("IsBadBuy", axis=1))
print("valid -", gini_score(res, val["IsBadBuy"]))
res = tree.predict_proba(test.drop("IsBadBuy", axis=1))
print("test  -", gini_score(res, test["IsBadBuy"]))

train - 0.9185672815846571
valid - 0.25484957761197347
test  - 0.2679247204692883


In [ ]:
xgb_res = xgb.predict_proba(train.drop("IsBadBuy", axis=1))
print("train -", gini_score(xgb_res[:, 1], train["IsBadBuy"]))
xgb_res = xgb.predict_proba(val.drop("IsBadBuy", axis=1))
print("valid -", gini_score(xgb_res[:, 1], val["IsBadBuy"]))
xgb_res = xgb.predict_proba(test.drop("IsBadBuy", axis=1))
print("test  -", gini_score(xgb_res[:, 1], test["IsBadBuy"]))

train - 0.9638497704412679
valid - 0.421802478391313
test  - 0.4114293870935306


In [ ]:
cbt_res = cbt.predict_proba(cat_train.drop("IsBadBuy", axis=1))
print("train -", gini_score(cbt_res[:, 1], train["IsBadBuy"]))
cbt_res = cbt.predict_proba(cat_val.drop("IsBadBuy", axis=1))
print("valid -", gini_score(cbt_res[:, 1], val["IsBadBuy"]))
cbt_res = cbt.predict_proba(cat_test.drop("IsBadBuy", axis=1))
print("test  -", gini_score(cbt_res[:, 1], test["IsBadBuy"]))

train - 0.7068716445047505
valid - 0.4388808045083852
test  - 0.4514621073358418


## 9.Extra Trees Classifier

See p.5